# XDMA BRAM + AXI-Lite Controller Test

This notebook tests two paths:

- PCIe XDMA memory-mapped DMA: `h2c_0/c2h_0` writes and reads the BRAM data window.
- AXI-Lite controller: `h2c_0/c2h_0` accesses the controller register window through the XDMA `M_AXI` address map, starts the RTL processing engine, polls status, then DMA reads back the processed BRAM output.

The current BD keeps the BRAM data window at `8K`: `[0x10000000, 0x10002000)`. The controller register window is `[0x10002000, 0x10003000)`.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import time
from dataclasses import dataclass

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "bin" / "xdma_rw.exe").exists():
    repo_candidate = Path("/home/triton/task/YOLO_on_FPGA/fpga/local/tests/xdma_bram")
    if (repo_candidate / "bin" / "xdma_rw.exe").exists():
        NOTEBOOK_DIR = repo_candidate

XDMA_EXE = NOTEBOOK_DIR / "bin" / "xdma_rw.exe"
DATA_DIR = NOTEBOOK_DIR / "data"
READBACK_DIR = NOTEBOOK_DIR / "readbacks"
REG_DIR = NOTEBOOK_DIR / "reg_io"
for path in (DATA_DIR, READBACK_DIR, REG_DIR):
    path.mkdir(exist_ok=True)

BRAM_BASE = 0x10000000
BRAM_SIZE = 0x2000
BRAM_WORD_BYTES = 32
CHANNEL = 0
AXIL_BASE = 0x10002000
CMD_TIMEOUT_SEC = 30

REG_CTRL = 0x00
REG_STATUS = 0x04
REG_SRC_ADDR = 0x08
REG_DST_ADDR = 0x0C
REG_LEN_BYTES = 0x10
REG_OP = 0x14
REG_OP_ARG = 0x18
REG_WORDS_DONE = 0x1C
REG_VERSION = 0x20

assert XDMA_EXE.exists(), f"Cannot find XDMA tool: {XDMA_EXE}"
print(f"XDMA tool: {XDMA_EXE}")
print(f"Test dir : {NOTEBOOK_DIR}")
print(f"BRAM     : 0x{BRAM_BASE:08X}..0x{BRAM_BASE + BRAM_SIZE:08X}")
print(f"AXI-Lite : 0x{AXIL_BASE:08X}..0x{AXIL_BASE + 0x1000:08X}")

In [ ]:
@dataclass
class CmdResult:
    cmd: list[str]
    returncode: int
    stdout: str
    stderr: str
    elapsed_sec: float

    @property
    def ok(self) -> bool:
        return self.returncode == 0


def run_cmd(cmd: list[str], timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    start = time.perf_counter()
    cp = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    return CmdResult(cmd, cp.returncode, cp.stdout, cp.stderr, time.perf_counter() - start)


def print_result(name: str, result: CmdResult) -> None:
    print(f"{name}: {'PASS' if result.ok else 'FAIL'}, ret={result.returncode}, elapsed={result.elapsed_sec:.3f}s")
    print("CMD:", " ".join(result.cmd))
    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())


def make_pattern(size: int, seed: int = 0) -> bytes:
    value = seed & 0xFFFFFFFF
    out = bytearray()
    for _ in range(size):
        value = (1664525 * value + 1013904223) & 0xFFFFFFFF
        out.append((value >> 24) & 0xFF)
    return bytes(out)


def validate_bram_range(offset: int, size: int, word_aligned: bool = False) -> None:
    assert offset >= 0
    assert size > 0
    assert offset + size <= BRAM_SIZE, f"range exceeds 8K BRAM window: offset=0x{offset:X}, size=0x{size:X}"
    if word_aligned:
        assert offset % BRAM_WORD_BYTES == 0, f"offset must be {BRAM_WORD_BYTES}-byte aligned"
        assert size % BRAM_WORD_BYTES == 0, f"size must be {BRAM_WORD_BYTES}-byte aligned"


def xdma_write(channel: int, addr: int, payload_path: Path) -> CmdResult:
    return run_cmd([str(XDMA_EXE), f"h2c_{channel}", "write", hex(addr), "-b", "-f", str(payload_path)])


def xdma_read(channel: int, addr: int, size: int, out_path: Path) -> CmdResult:
    if out_path.exists():
        out_path.unlink()
    return run_cmd([str(XDMA_EXE), f"c2h_{channel}", "read", hex(addr), "-l", hex(size), "-b", "-f", str(out_path)])


def compare_bytes(expected: bytes, actual_path: Path, max_diff: int = 16) -> tuple[bool, str]:
    if not actual_path.exists():
        return False, f"missing readback file: {actual_path}"
    actual = actual_path.read_bytes()
    if expected == actual:
        digest = hashlib.sha256(actual).hexdigest()
        return True, f"match: {len(actual)} bytes, sha256={digest}"
    lines = [f"mismatch: expected_len={len(expected)}, actual_len={len(actual)}"]
    for index, (a, b) in enumerate(zip(expected, actual)):
        if a != b:
            lines.append(f"0x{index:08X}: expected=0x{a:02X}, actual=0x{b:02X}")
            if len(lines) > max_diff:
                break
    return False, "\n".join(lines)


def axil_write_u32(offset: int, value: int) -> CmdResult:
    path = REG_DIR / f"wr_{offset:03X}.bin"
    path.write_bytes((value & 0xFFFFFFFF).to_bytes(4, "little"))
    return xdma_write(CHANNEL, AXIL_BASE + offset, path)


def axil_read_u32(offset: int) -> tuple[int, CmdResult]:
    path = REG_DIR / f"rd_{offset:03X}.bin"
    if path.exists():
        path.unlink()
    result = xdma_read(CHANNEL, AXIL_BASE + offset, 4, path)
    value = int.from_bytes(path.read_bytes()[:4], "little") if result.ok and path.exists() else 0
    return value, result


def golden_process(payload: bytes, op: int, arg: int) -> bytes:
    assert len(payload) % 4 == 0
    out = bytearray()
    for i in range(0, len(payload), 4):
        lane = int.from_bytes(payload[i:i+4], "little")
        if op == 0:
            value = lane
        elif op == 1:
            value = lane ^ arg
        elif op == 2:
            value = (lane + arg) & 0xFFFFFFFF
        elif op == 3:
            value = (~lane) & 0xFFFFFFFF
        else:
            raise ValueError(op)
        out += value.to_bytes(4, "little")
    return bytes(out)


def poll_done(timeout_sec: float = 2.0) -> dict:
    deadline = time.time() + timeout_sec
    last_status = 0
    while time.time() < deadline:
        status, rd = axil_read_u32(REG_STATUS)
        if not rd.ok:
            print_result("AXIL READ STATUS", rd)
            return {"ok": False, "status": status, "detail": "status read failed"}
        last_status = status
        busy = bool(status & 0x1)
        done = bool(status & 0x2)
        error = bool(status & 0x4)
        if done or error:
            return {"ok": done and not error, "status": status, "busy": busy, "done": done, "error": error}
        time.sleep(0.01)
    return {"ok": False, "status": last_status, "detail": "timeout"}


In [ ]:
def run_dma_rw_case(offset: int, size: int, seed: int = 0) -> dict:
    validate_bram_range(offset, size)
    payload = make_pattern(size, seed)
    write_path = DATA_DIR / f"dma_write_off{offset:04X}_size{size:04X}.bin"
    read_path = READBACK_DIR / f"dma_read_off{offset:04X}_size{size:04X}.bin"
    write_path.write_bytes(payload)

    wr = xdma_write(CHANNEL, BRAM_BASE + offset, write_path)
    print_result("DMA WRITE", wr)
    rd = xdma_read(CHANNEL, BRAM_BASE + offset, size, read_path)
    print_result("DMA READ", rd)
    match, detail = compare_bytes(payload, read_path)
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    return {"pass": wr.ok and rd.ok and match, "write_ok": wr.ok, "read_ok": rd.ok, "match": match, "detail": detail}


dma_results = [
    run_dma_rw_case(0x0000, 0x0040, seed=0x11),
    run_dma_rw_case(0x0400, 0x0400, seed=0x22),
    run_dma_rw_case(0x1800, 0x0800, seed=0x33),
]
assert all(item["pass"] for item in dma_results), "DMA BRAM read/write test failed"
dma_results

In [ ]:
# Probe AXI-Lite register access. VERSION should be 0x58424101.
version, version_rd = axil_read_u32(REG_VERSION)
print_result("AXIL READ VERSION", version_rd)
print(f"VERSION = 0x{version:08X}")
assert version_rd.ok, "AXI-Lite register read failed"
assert version == 0x58424101, f"unexpected controller VERSION: 0x{version:08X}"

status, status_rd = axil_read_u32(REG_STATUS)
print_result("AXIL READ STATUS", status_rd)
print(f"STATUS = 0x{status:08X}")
assert status_rd.ok

In [ ]:
def run_controller_case(src_offset: int, dst_offset: int, size: int, op: int, arg: int, seed: int = 0) -> dict:
    validate_bram_range(src_offset, size, word_aligned=True)
    validate_bram_range(dst_offset, size, word_aligned=True)
    assert src_offset + size <= dst_offset or dst_offset + size <= src_offset, "source and destination regions should not overlap"

    payload = make_pattern(size, seed)
    expected = golden_process(payload, op, arg)
    input_path = DATA_DIR / f"ctrl_input_src{src_offset:04X}_size{size:04X}.bin"
    output_path = READBACK_DIR / f"ctrl_output_dst{dst_offset:04X}_size{size:04X}_op{op}.bin"
    input_path.write_bytes(payload)

    wr = xdma_write(CHANNEL, BRAM_BASE + src_offset, input_path)
    print_result("DMA WRITE INPUT", wr)
    assert wr.ok, "input DMA write failed"

    for name, offset, value in [
        ("SRC_ADDR", REG_SRC_ADDR, src_offset),
        ("DST_ADDR", REG_DST_ADDR, dst_offset),
        ("LEN_BYTES", REG_LEN_BYTES, size),
        ("OP", REG_OP, op),
        ("OP_ARG", REG_OP_ARG, arg),
    ]:
        result = axil_write_u32(offset, value)
        print_result(f"AXIL WRITE {name}", result)
        assert result.ok, f"AXI-Lite write {name} failed"

    start = axil_write_u32(REG_CTRL, 1)
    print_result("AXIL WRITE CTRL.start", start)
    assert start.ok, "AXI-Lite start failed"

    poll = poll_done(timeout_sec=2.0)
    print("POLL:", poll)
    assert poll["ok"], f"controller did not complete cleanly: {poll}"

    words_done, words_done_rd = axil_read_u32(REG_WORDS_DONE)
    print_result("AXIL READ WORDS_DONE", words_done_rd)
    print(f"WORDS_DONE = {words_done}")
    assert words_done == size // BRAM_WORD_BYTES, "unexpected WORDS_DONE"

    rd = xdma_read(CHANNEL, BRAM_BASE + dst_offset, size, output_path)
    print_result("DMA READ OUTPUT", rd)
    match, detail = compare_bytes(expected, output_path)
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    return {"pass": rd.ok and match, "read_ok": rd.ok, "match": match, "detail": detail, "poll": poll}


controller_results = [
    run_controller_case(src_offset=0x0000, dst_offset=0x1000, size=0x0400, op=0, arg=0x00000000, seed=0x100),
    run_controller_case(src_offset=0x0000, dst_offset=0x1000, size=0x0400, op=1, arg=0xA5A5A5A5, seed=0x101),
    run_controller_case(src_offset=0x0000, dst_offset=0x1000, size=0x0400, op=2, arg=0x00000001, seed=0x102),
    run_controller_case(src_offset=0x0000, dst_offset=0x1000, size=0x0400, op=3, arg=0x00000000, seed=0x103),
]
assert all(item["pass"] for item in controller_results), "AXI-Lite controlled BRAM processing test failed"
controller_results